# Zenith quest-breakdown model — full training run (Colab)

**Before running anything:** `Runtime -> Change runtime type -> T4 GPU`.

This clones the public repo, installs `ml/`'s requirements, runs the real
full training config (~8 epochs over 4,800 examples, fp32 — a T4 does this
in well under an hour), evaluates against the acceptance criteria, and
converts + downloads the ONNX model if it passes. Local Apple Silicon (MPS)
was tried first and works correctly, but on an 8GB machine it thrashes into
swap and projects ~12 hours for this run — a real GPU is the practical path.

See `ml/README.md` in the repo for full pipeline context. Run cells top to
bottom; read the markdown cell after the eval step before continuing.

In [ ]:
import torch
print("GPU available:", torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU — set Runtime > Change runtime type > T4 GPU, then re-run this cell")
assert torch.cuda.is_available(), "No GPU detected — change the runtime type before continuing."

In [ ]:
!git clone --depth 1 https://github.com/coderxdk-1812/quest-tracker.git
%cd quest-tracker/ml

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
# The dataset is already committed to the repo (250 hand-authored scenarios,
# 0 validation failures — see ml/README.md's "Dataset" section) — no need to
# regenerate it. This just confirms it came through the clone intact.
!wc -l data/processed/*.jsonl

In [ ]:
# Full training run. fp16 is deliberately OFF (T5 diverges to NaN in fp16 —
# train.py's guard_t5_fp16 enforces this regardless of the config), so this
# runs in fp32 — correct and still fast on a T4 at these batch sizes.
!python3 train.py --config configs/full.json

In [ ]:
!python3 eval.py --model_dir runs/full --test_file data/processed/test.jsonl

## Check the acceptance table printed above before continuing

- JSON-validity rate ≥ 98%
- Rubric pass rate ≥ 95%

If either misses the bar, open `runs/full/eval_report.json` and look at its
`summary.by_intent` / `summary.by_task_type` breakdown (you can `!cat` it or
download it via the file browser on the left):

- **One or two intents/task types consistently weak, rest fine** — that's a
  data problem. Add more genuinely distinct scenarios to
  `data/scenarios/<intent>.py` (not more augmentation — see `ml/README.md`'s
  "Dataset" section for why 20 templates × 220 clones failed this exact way
  the first time), rerun `build_dataset.py`, and re-run the training cell
  above with a fresh `output_dir` (e.g. edit `configs/full.json`'s
  `output_dir` to `runs/full_v2`, or just rerun — old checkpoints aren't
  clobbered as long as the dir differs).
- **Broadly weak everywhere, even though the dataset is genuinely diverse**
  — that's a capacity problem, not a data problem. Re-run the training cell
  with `configs/full_base.json` instead (scales to `flan-t5-base`) rather
  than adding more data.

Only run the conversion cells below once both criteria pass.

In [ ]:
!python3 convert_to_onnx.py --model_dir runs/full --out_dir onnx/quest-model

In [ ]:
# Zip and download the browser-ready (ONNX, int8-quantized) model files.
!zip -rq quest-model-onnx.zip onnx/quest-model
from google.colab import files
files.download("quest-model-onnx.zip")

In [ ]:
# Optional: also grab the raw PyTorch checkpoint, in case you want to
# re-evaluate, re-convert, or resume training later without rerunning all
# 8 epochs. Skips optimizer/scheduler/rng state to keep the download small
# (fine — you're not resuming mid-epoch, just reusing the trained weights).
!zip -rq runs-full-checkpoint.zip runs/full -x "runs/full/checkpoint-*/optimizer.pt" "runs/full/checkpoint-*/scheduler.pt" "runs/full/checkpoint-*/rng_state.pth"
from google.colab import files
files.download("runs-full-checkpoint.zip")

## Next steps (back on your machine, or anywhere with git)

1. Unzip `quest-model-onnx.zip` into `ml/onnx/quest-model/` in your local
   clone.
2. Host it — HF Hub (recommended, one `huggingface-cli upload` command) or
   `public/models/` with Git LFS. See `ml/README.md`'s "Host it" section
   for both options, with exact commands.
3. Point `MODEL_ID` in `src/lib/localModel.ts` at the hosted repo/path —
   it's currently a placeholder (`zenith-app/quest-breakdown-flan-t5-small`).
   That one-line edit is the entire remaining integration step; everything
   else (Settings toggle, download-progress UI, rule-engine fallback) is
   already wired.
4. `npm run build` and verify in the browser: enable the on-device
   assistant in Settings, generate a task breakdown, confirm it downloads
   and uses the real model instead of falling back to the rule engine.